<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🌊 M3 Lab 2 · 深度研究 Flow + 可观测性

**一句话**：把 Lab 1 的研究 crew 包进一个 CrewAI **Flow**：先让 LLM 判断问题难不难，简单的直接答，复杂的才动用整个 crew；再用 tracing 看每一步。

<span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🏷️ Lab</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">📄 C1M3_Lab_2_deep_research_flow.ipynb</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">▶️ cd deep_research_flow &amp;&amp; crewai run</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🧩 前置：M3 Lab 1 并行研究 crew</span>

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**要解决的问题**：不是每个问题都值得跑几分钟、花一堆搜索费的研究 crew。「Hello!」直接答就好。
本 lab 用 Flow 做**编排**：
- `@start` 读用户问题 → `@router` 用一次 LLM 调用判断 SIMPLE / RESEARCH；
- SIMPLE → 一次 LLM 调用直接回答；RESEARCH → 追问澄清 → 跑 crew → 存报告 + 摘要；
- 两条路汇合到最终回答；`@persist` 把状态存进 SQLite，`tracing=True` 生成可视化追踪。

**📝 TODO 状态**：`main.py` 里的空位原本**全是 `None`**（你还没填）。已用 `Solution/main.py` 填好（两份文件只差这些空位）。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧠 Flow 长什么样

</div>

<img src="../03_building-coordination-patterns/images/m3-lab1-agents-tasks-diagram.png" width="520">  <img src="../03_building-coordination-patterns/images/m3-lab2-flow-diagram.png" width="380">

<div style="background:#111827; border:1px dashed #334155; border-radius:8px; padding:10px 16px; width:97%;"><div style="color:#94a3b8; font-weight:700; margin-bottom:6px;">① 简单问题</div><span style="display:inline-block; background:#0f172a; border:1.5px solid #a78bfa; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">start_conversation</code> @start</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #fbbf24; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">analyze_query</code> @router</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #60a5fa; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">"SIMPLE" → <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">simple_answer</code></span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #4ade80; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">return_final_answer</code></span></div>

<div style="background:#111827; border:1px dashed #334155; border-radius:8px; padding:10px 16px; width:97%;"><div style="color:#94a3b8; font-weight:700; margin-bottom:6px;">② 复杂问题</div><span style="display:inline-block; background:#0f172a; border:1.5px solid #fbbf24; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">analyze_query</code> @router</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #f87171; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">"RESEARCH" → <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">clarify_query</code></span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #f87171; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">execute_research</code>（跑 crew）</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #f87171; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">save_report_and_summarize</code></span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #4ade80; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">return_final_answer</code> = or_(…)</span></div>

<div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(260px,1fr)); gap:10px; width:97%;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #a78bfa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047;">Crew（Lab 1）</div><ul style="margin:6px 0 0 0; padding-left:18px; font-size:0.92em; line-height:1.6;"><li style="margin:2px 0;">一群 agent 自主协作完成一组 task</li><li style="margin:2px 0;">流程由 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">process</code> + task 顺序 / context 决定</li><li style="margin:2px 0;">适合「怎么做」交给 LLM 的开放任务</li></ul></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #4ade80; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047;">Flow（本 lab）</div><ul style="margin:6px 0 0 0; padding-left:18px; font-size:0.92em; line-height:1.6;"><li style="margin:2px 0;"><b>你写的 Python 方法</b>按事件连起来（事件驱动）</li><li style="margin:2px 0;">分支、循环、状态都是确定的代码</li><li style="margin:2px 0;">crew 只是其中一个步骤；能省就省</li></ul></div></div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧠 四个装饰器 + state

</div>

<div style="width:97%;"><div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(230px,1fr)); gap:10px;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #a78bfa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🚀 @start()</div><div style="font-size:0.92em; line-height:1.6;">入口方法，<code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">kickoff()</code> 后最先运行（可以有多个，会并发）。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #60a5fa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">👂 @listen(x)</div><div style="font-size:0.92em; line-height:1.6;">x 完成后运行。x 可以是<b>方法</b>（对象或名字字符串），也可以是 router 返回的<b>字符串</b>（如 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">&quot;SIMPLE&quot;</code>）。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #fbbf24; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🔀 @router(x)</div><div style="font-size:0.92em; line-height:1.6;">x 完成后运行；它的<b>返回值</b>是一个事件名，只有 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">@listen(&quot;那个值&quot;)</code> 的方法会被触发。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #4ade80; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🔗 or_(a, b)</div><div style="font-size:0.92em; line-height:1.6;">a 或 b 任一完成就触发；<code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">and_(a, b)</code> 则要都完成。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #f87171; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">💾 @persist()</div><div style="font-size:0.92em; line-height:1.6;">加在<b>类</b>上：每个方法跑完都把 state 存进 SQLite；<code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">kickoff(inputs={&quot;id&quot;: …})</code> 用同一 id 时会<b>先恢复上次的 state</b>。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #22d3ee; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🧾 Flow[ResearchState]</div><div style="font-size:0.92em; line-height:1.6;">state 是 pydantic 模型，方法里用 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">self.state.xxx</code> 读写；字段都要有默认值（Flow 自己创建实例）。</div></div></div></div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🗂️ 项目结构

</div>

和 Lab 1 相似，但这是 `crewai create flow deep_research_flow` 生成的 **flow 项目**：crew 被放进 `src/deep_research_flow/crews/` 里，`main.py` 定义 Flow。

<img src="../03_building-coordination-patterns/images/lab2-folder-structure.png" width="200">

<div style="width:97%;"><div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(230px,1fr)); gap:10px;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #60a5fa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">📄 pyproject.toml</div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">[tool.crewai] type = &quot;flow&quot;</code>；<code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">kickoff</code> / <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">run_crew</code> 都指向 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">main:kickoff</code>，<code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">plot</code> 指向 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">main:plot</code>。钉死 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">crewai[tools]==1.3.0</code>。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #a78bfa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">👥 crews/deep_research_crew/</div><div style="font-size:0.92em; line-height:1.6;">Lab 1 的 crew，<b>去掉了 after-kickoff hook</b>（存文件交给 Flow）。注意它用的是正确的 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">markdown=True</code>，输出文件叫 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">report.md</code>。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #4ade80; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🌊 main.py</div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">ResearchState</code> + <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">DeepResearchFlow</code> + <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">kickoff()</code> / <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">plot()</code>，本 lab 要填的全在这里。</div></div></div></div>

In [1]:
import os, sys
from pathlib import Path

# VSCode 的 kernel PATH 不一定含 .venv/bin → 放到最前，`!crewai` / `uv` 才找得到
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"
os.environ.pop("VIRTUAL_ENV", None)   # 项目用自己的 .venv（crewai 1.3.0），免得 uv 报 VIRTUAL_ENV 不匹配警告

PROJ = Path("deep_research_flow")
SRC = PROJ / "src" / "deep_research_flow"
MAIN = (SRC / "main.py").read_text()

def between(start, end=None):
    """打印 main.py 中从 start 所在行到 end 所在行之前的片段"""
    a = MAIN.rfind("\n", 0, MAIN.index(start)) + 1
    b = MAIN.rfind("\n", 0, MAIN.index(end, a)) + 1 if end else len(MAIN)
    print(MAIN[a:b].rstrip())

for p in sorted(PROJ.rglob("*")):
    if p.is_file() and not {".venv", "__pycache__"} & set(p.parts) and p.name != ".DS_Store":
        print(p)

deep_research_flow/__init__.py
deep_research_flow/knowledge/user_preference.txt
deep_research_flow/pyproject.toml
deep_research_flow/report.md
deep_research_flow/src/deep_research_flow/__init__.py
deep_research_flow/src/deep_research_flow/crews/deep_research_crew/__init__.py
deep_research_flow/src/deep_research_flow/crews/deep_research_crew/config/agents.yaml
deep_research_flow/src/deep_research_flow/crews/deep_research_crew/config/tasks.yaml
deep_research_flow/src/deep_research_flow/crews/deep_research_crew/crew.py
deep_research_flow/src/deep_research_flow/crews/deep_research_crew/guardrails/__init__.py
deep_research_flow/src/deep_research_flow/crews/deep_research_crew/guardrails/guardrails.py
deep_research_flow/src/deep_research_flow/main.py
deep_research_flow/src/deep_research_flow/tools/__init__.py
deep_research_flow/src/deep_research_flow/utils.py
deep_research_flow/uv.lock


In [2]:
# run this cell to avoid warnings
import warnings
warnings.filterwarnings("ignore")

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 3.1 state：Flow 的「记事本」

</div>

四个字段：`user_query`（问题）、`needs_research`（是否走研究）、`research_report`（crew 报告）、`final_answer`（最终回答）。类型 + 默认值都要写。

In [3]:
between("# define the flow state", "### START CODE HERE ###\n# add persistence")

# define the flow state
class ResearchState(BaseModel):
    user_query: str = ""
    ### START CODE HERE ###
    needs_research: bool = False
    research_report: str = ""
    final_answer: str = ""
    ### END CODE HERE ###


<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 3.2–3.3 @persist 与 @start 入口

</div>

`@persist()` 放在 **class 上**（所有方法跑完都存档）；`start_conversation` 用 `@start()` 标为入口：打印上次记住的问题，再用 `input()` 读新问题。

In [4]:
between("### START CODE HERE ###\n# add persistence", "# define the router")

### START CODE HERE ###
# add persistence to the flow
@persist()
### END CODE HERE ###

class DeepResearchFlow(Flow[ResearchState]):
    # define the entrypoint
    ### START CODE HERE ###
    @start()
    ### END CODE HERE ###
    def start_conversation(self):
        """Entry point for the flow"""
        print("🔍 Deep Research Flow started")
        print(self.state.user_query)
        if self.state.user_query != "":
            print(f"I remember last time you wanted to know about {self.state.user_query}")
        self.state.user_query = input("\033 What would you like to know?\033  \n>> \n")
        print(f"Query received: \"{self.state.user_query}\"")


<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**入口用了 `input()`**：所以课程说这个 flow「不能在 Jupyter 里跑」。本 notebook 的解决办法：在 shell 里用管道把答案喂给 stdin（`printf '问题\n' | crewai run`），不用改项目代码。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 3.4 @router：要不要研究？

</div>

填两处：装饰器 `@router(start_conversation)`；prompt 里写清楚**只回一个词** SIMPLE 或 RESEARCH 及各自标准。
返回值 `"RESEARCH"` / `"SIMPLE"` 就是事件名。这里用 `"RESEARCH" in decision.upper()` 判断，对模型多说几个字也能容错。

In [5]:
between("# define the router", "# define the simple answer task")

    # define the router
    ### START CODE HERE ###
    @router(start_conversation)
    ### END CODE HERE ###
    def analyze_query(self):
        """Router: Should trigger research?"""
        print("🤔 Analyzing query complexity...")
        
        ### START CODE HERE ###
        prompt = (# Write the prompt for the LLM to decide if the query is simple or requires research
                  ""
                  "Analyze this query and respond with exactly one word: SIMPLE or RESEARCH\n\n"
                  "SIMPLE: greetings, basic questions, well-known facts, context-based queries\n"
                  "RESEARCH: complex topics requiring comprehensive investigation, current events, detailed analysis, multi-faceted questions\n\n"
                  f"Query: \"{self.state.user_query}\"\n\n"
                  "Response (one word only):")
        ### END CODE HERE ###

        # define the llm for the decision 
        llm = LLM(model="gpt-4o-mini",)
        # call the llm and save the r

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 3.5 SIMPLE 分支：一次 LLM 调用

</div>

`@listen("SIMPLE")` 已给出。填：prompt 前半句、`LLM(model="gpt-4o-mini")`、`llm.call(messages=prompt)`，结果写入 `self.state.final_answer`。`llm.call` 接受字符串，会自动包成一条 user 消息。

In [6]:
between("# define the simple answer task", "# define the clarification task")

    # define the simple answer task (no research needed)
    @listen("SIMPLE")
    def simple_answer(self):
        """LLM: Direct answer for simple queries"""
        print("✨ Generating direct answer...")
        
        ### START CODE HERE ###
        prompt = (# Write the missing part of the query for the LLM
                 ""
                 "Provide a direct, helpful, and comprehensive answer to this query. "
                 "Be informative but concise.\n\n"
                 f"Query: \"{self.state.user_query}\"\n\n"
                 "Answer:"
                 )
        # set up the LLM
        llm = LLM(model="gpt-4o-mini",)
        # call the llm with the prompt and save the result to the final_answer state variable
        self.state.final_answer = llm.call(messages=prompt)
        ### END CODE HERE ###


<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 3.6 RESEARCH 分支：澄清 → crew → 存档摘要

</div>

三个方法用 `@listen` 串成一条链：
- `clarify_query`：`@listen("RESEARCH")`；LLM 回 PROCEED 或 `CLARIFY: 问题`，需要时再 `input()` 一次，把补充信息追加到 `user_query`；
- `execute_research`：`@listen("clarify_query")`；`crew().kickoff(inputs={"user_query": self.state.user_query})`，存 `result.raw`；
- `save_report_and_summarize`：`@listen(execute_research)`（传方法对象也行）；写 `../research_report.md`，再让 LLM 压成一段摘要。

In [7]:
between("# define the clarification task", "# define the final answer task")

    # define the clarification task (if research is needed)
    ### START CODE HERE ###
    @listen("RESEARCH")
    #### END CODE HERE ###
    def clarify_query(self):
        """LLM: Clarification before research"""
        print("🔍 Reviewing query for research clarity...")
        
        # write the prompt to decide if the query is clear enough
        prompt = ("Review this research query and determine if it's clear enough "
                 "for comprehensive research.\n\n"
                 "Respond in one of these formats:\n"
                 "- If clear and specific: \"PROCEED\"\n"
                 "- If needs clarification: \"CLARIFY: [specific questions to ask the user]\"\n\n"
                 f"Query: \"{self.state.user_query}\"\n\n"
                 "Response:"
                 )
        # define the llm and call it with the prompt
        llm = LLM(model="gpt-4o-mini",)
        response = llm.call(messages=prompt)

        # if the query is not clear, ask the user for clar

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 3.7–3.8 汇合 + tracing

</div>

`return_final_answer` 用 `@listen(or_("simple_answer", "save_report_and_summarize"))`：哪条路先完成都触发。`kickoff()` 里填 `DeepResearchFlow(tracing=True)`，运行结束会打印 CrewAI 平台上的追踪链接。

In [8]:
between("# define the final answer task")

    # define the final answer task
    @listen(or_("simple_answer", "save_report_and_summarize"))
    def return_final_answer(self):
        """Return the final answer to the user"""
        print("📝 Final Answer:")
        print(f"📌 Original Query: \"{self.state.user_query}\"")
        print(f"{self.state.final_answer}")
        print("\n✨ Deep Research Flow completed!")

    

def kickoff():
    ### START CODE HERE ###
    # instantiate the DeepResearchFlow with tracing enabled
    deep_research_flow = DeepResearchFlow(tracing=True)
    ### END CODE HERE ###
    
    # kickoff the flow with a custom id, so you can persist the state
    deep_research_flow.kickoff(inputs={"id": "our-deep-research_flow"})
    

def plot():
    deep_research_flow = DeepResearchFlow()
    deep_research_flow.plot()


if __name__ == "__main__":
    kickoff()


<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**`kickoff(inputs={"id": "our-deep-research_flow"})` 的 `id`**：配合 `@persist`，固定 id = 每次运行都接着同一份 state。所以第二次运行会打印「I remember last time you wanted to know about …」。
**同步 vs 异步**：项目里是 `flow.kickoff()`（内部 `asyncio.run`），适合终端；在 notebook 里直接调 Flow 要用 `await flow.kickoff_async(...)`，否则撞上 Jupyter 已在运行的事件循环。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🗺️ 4. 画出 Flow

</div>

`crewai flow plot` 执行 `uv run plot` → `DeepResearchFlow().plot()`。第一次会先建项目的 `.venv`（一两分钟）。

In [9]:
! cd deep_research_flow && crewai flow plot

Plotting the Flow


╭─────────────────────────────── Flow Execution ───────────────────────────────╮
│                                                                              │
│  Starting Flow Execution                                                     │
│  Name: DeepResearchFlow                                                      │
│  ID: bf981a11-4693-40ea-97f8-a45d1fe0e582                                    │
│  Tool Args:                                                                  │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯



Trace batch initialization returned status 401. Continuing without tracing.


<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🆕 2026-10 版本差异**：课程说 HTML 会生成在 `deep_research_flow/` 里——那是旧版（0.x）行为。crewai **1.3.0** 的 `plot()` 把 `crewai_flow.html` 写进**临时目录**（`tempfile.mkdtemp(prefix="crewai_flow_")`）并**自动用浏览器打开**（源码 `flow/visualization/renderers/interactive.py`）。所以项目里找不到这个文件，去弹出的浏览器标签页看。
输出里的 `Trace batch initialization returned status 401` 无害：plot 时没开 tracing 账号，跳过追踪而已。

</div>

在 kernel 里也能拿到同一份图结构（kernel 的 crewai 1.15.22，直接 import 项目代码，不运行 flow）：

In [10]:
sys.path.insert(0, str(SRC.parent))   # 让 `deep_research_flow` 包可被 import
from deep_research_flow.main import DeepResearchFlow
from crewai.flow.visualization.builder import build_flow_structure

g = build_flow_structure(DeepResearchFlow)
for name, node in g["nodes"].items():
    print(f"{node['type']:7} {name}")
print()
for e in g["edges"]:
    print(f"{e['source']} → {e['target']}")

Router events for 'analyze_query' are dynamic or not statically inferable; static visualization may omit event edges.


Static visualization could not match listener triggers {'SIMPLE', 'RESEARCH'} to explicit router events. Dynamic router values may still trigger these listeners at runtime.


start   start_conversation
router  analyze_query
listen  simple_answer
listen  clarify_query
listen  execute_research
listen  save_report_and_summarize
listen  return_final_answer

start_conversation → analyze_query
clarify_query → execute_research
execute_research → save_report_and_summarize
simple_answer → return_final_answer
save_report_and_summarize → return_final_answer


<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**少了两条边？** `analyze_query → simple_answer / clarify_query` 没画出来，日志也提示 *Router events … are dynamic or not statically inferable*：router 的返回值写在 if/else 里，静态分析读不出来。运行时完全正常。
🆕 kernel 里的 crewai 1.15.22 可以补上：`@router(start_conversation, emit=["SIMPLE", "RESEARCH"])`，或把返回类型标成 `Literal["SIMPLE", "RESEARCH"]`（见 `crewai/flow/dsl/_router.py` 文档）；项目钉的 1.3.0 的 `router()` 只有 `condition` 一个参数。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🚀 5. 运行 Flow

</div>

课程让你在终端里跑 `crewai run` 并手动输入问题。这里用 `printf … |` 把答案提前写进 stdin，每一行喂给一次 `input()`。
**先跑一个简单问题**：只有 3 次 LLM 调用，几秒钟。

In [11]:
! cd deep_research_flow && printf 'Hello! What is CrewAI in one sentence?\n' | crewai run

╭─────────────────────────────── Flow Execution ───────────────────────────────╮
│                                                                              │
│  Starting Flow Execution                                                     │
│  Name: DeepResearchFlow                                                      │
│  ID: 1fa17d88-6e91-4231-a842-cabf76c2012e                                    │
│  Tool Args:                                                                  │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

Loading flow state from memory for UUID: our-deep-research_flow
Loading flow state from memory for UUID: our-deep-research_flow
Flow started with ID: our-deep-research_flow
Flow started with ID: our-deep-research_flow
🔍 Deep Research Flow started
Hello! What is CrewAI i

🌊 Flow: DeepResearchFlow
ID: 1fa17d88-6e91-4231-a842-cabf76c2012e
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
└── 🔄 Running: analyze_query

🌊 Flow: DeepResearchFlow
ID: 1fa17d88-6e91-4231-a842-cabf76c2012e
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
└── 🔄 Running: analyze_query

💬 Simple query detected - providing direct answer
🌊 Flow: DeepResearchFlow
ID: 1fa17d88-6e91-4231-a842-cabf76c2012e
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
🌊 Flow: DeepResearchFlow
ID: 1fa17d88-6e91-4231-a842-cabf76c2012e
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
✨ Generating direct answer...
🌊 Flow: DeepResearchFlow
ID: 1fa17d88-6e91-4231-a842-cabf76c2012e
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
🌊 Flow: DeepResearchFlow
ID: 1fa17d88-6e91-4231-a842-cabf76c2012e
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
├── ✅ Completed: analyze_query
└── 🔄 Running: simple_answer

🌊 Flow: DeepResearchFlow
ID: 1fa17d88-6e91-4231-a842-cabf76c2012e
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
├── ✅ Completed: analyze_query
└── 🔄 Running: simple_answer

🌊 Flow: DeepResearchFlow
ID: 1fa17d88-6e91-4231-a842-cabf76c2012e
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
├── ✅ Completed: analyze_query
└── 🔄 Running: simple_answer

🌊 Flow: DeepResearchFlow
ID: 1fa17d88-6e91-4231-a842-cabf76c2012e
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
├── ✅ Completed: analyze_query
└── 🔄 Running: simple_answer

🌊 Flow: DeepResearchFlow
ID: 1fa17d88-6e91-4231-a842-cabf76c2012e
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
├── ✅ Completed: analyze_query
└── 🔄 Running: simple_answer

📝 Final Answer:
🌊 Flow: DeepResearchFlow
ID: 1fa17d88-6e91-4231-a842-cabf76c2012e
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
├── ✅ Completed: analyze_query
🌊 Flow: DeepResearchFlow
ID: 1fa17d88-6e91-4231-a842-cabf76c2012e
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
├── ✅ Completed: analyze_query
├── ✅ Completed: simple_answer
📌 Original Query: "Hello! What is CrewAI in one sentence?"
🌊 Flow: DeepResearchFlow
ID: 1fa17d88-6e91-4231-a842-cabf76c2012e
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
├── ✅ Completed: analyze_query
├── ✅ Completed: simple_answer
🌊 Flow: DeepResearchFlow
ID: 1fa17d88-6e91-4231-a842-cabf76c2012e
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
├── ✅ Completed: analyze_query
├── ✅ Completed: simple_answer
CrewAI is an advanced artificial intelligence platform designed to optimize team
collaboration and productivity by providing

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 7d71d2a1-a5de-4c20-9677-4c55a87a9bad                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/7d71d2a1-a5de-4c2 │
│ 0-9677-4c55a87a9bad?access_code=TRACE-ec649ffd62                             │
│ 🔑 Access Code: TRACE-ec649ffd62                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**看看 SIMPLE 分支的答案**：大概率把 CrewAI 说成「团队协作平台」之类——简单分支**没有搜索**，只靠模型记忆，会自信地编。这正是 router 标准要写清「需要最新信息 → RESEARCH」的原因。

</div>

**再跑一个研究问题**（约几分钟，会真的搜索、花钱）。第 1 行是问题；第 2 行是给 `clarify_query` 的补充说明——如果模型直接 PROCEED，这一行就不会被读取。
注意开头会打印 `I remember last time …`：这就是 `@persist` + 固定 id 恢复出来的上一次 state。

In [12]:
! cd deep_research_flow && printf 'Compare CrewAI, LangGraph and AutoGen for building production multi-agent systems: features, limitations and ideal use cases.\nFocus on Python developers in 2026; keep the report concise.\n' | crewai run

╭─────────────────────────────── Flow Execution ───────────────────────────────╮
│                                                                              │
│  Starting Flow Execution                                                     │
│  Name: DeepResearchFlow                                                      │
│  ID: 75b49323-85e0-48f6-96a2-062b9619db60                                    │
│  Tool Args:                                                                  │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯
Loading flow state from memory for UUID: our-deep-research_flow
Loading flow state from memory for UUID: our-deep-research_flow

Flow started with ID: our-deep-research_flow
Flow started with ID: our-deep-research_flow
🔍 Deep Research Flow started
Hello! What is CrewAI i

🌊 Flow: DeepResearchFlow
ID: 75b49323-85e0-48f6-96a2-062b9619db60
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
└── 🔄 Running: analyze_query

🌊 Flow: DeepResearchFlow
ID: 75b49323-85e0-48f6-96a2-062b9619db60
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
└── 🔄 Running: analyze_query

📚 Complex query detected - initiating research process
🌊 Flow: DeepResearchFlow
ID: 75b49323-85e0-48f6-96a2-062b9619db60
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
🌊 Flow: DeepResearchFlow
ID: 75b49323-85e0-48f6-96a2-062b9619db60
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
🔍 Reviewing query for research clarity...
🌊 Flow: DeepResearchFlow
ID: 75b49323-85e0-48f6-96a2-062b9619db60
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
🌊 Flow: DeepResearchFlow
ID: 75b49323-85e0-48f6-96a2-062b9619db60
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
├── ✅ Completed: analyze_query
└── 🔄 Running: clarify_query

🌊 Flow: DeepResearchFlow
ID: 75b49323-85e0-48f6-96a2-062b9619db60
├── ✨ Created
├── ✅ Initialization Complete
├── ✅ Completed: start_conversation
├── ✅ Completed: analyze_query
└── 🔄 Running: clarify_query

...

📋 Task: create_research_plan (ID: 147e0b33-2b5f-416d-ad66-9f41d69cbf49)
Status: Executing Task...
└── 🧠 Memory Update Overall
    ├── ✅ Short Term Memory Memory Saved (246.47ms)
    └── ✅ Long Term Memory Memory Saved (1.89ms)

📋 Task: create_research_plan (ID: 147e0b33-2b5f-416d-ad66-9f41d69cbf49)
Status: Executing Task...
└── 🧠 Memory Update Overall
    ├── ✅ Short Term Memory Memory Saved (246.47ms)
    └── ✅ Long Term Memory Memory Saved (1.89ms)

📋 Task: create_research_plan (ID: 147e0b33-2b5f-416d-ad66-9f41d69cbf49)
Status: Executing Task...
└── 🧠 Memory Update Overall
    ├── ✅ Short Term Memory Memory Saved (246.47ms)
    └── ✅ Long Term Memory Memory Saved (1.89ms)

🚀 Crew: crew
├── ✅ Memory Retrieval Completed
│   └── Sources Used
│       ├── ✅ Long Term Memory (0.79ms)
│       └── ✅ Short Term Memory (168.99ms)
├── 📋 Task: create_research_plan (ID: 147e0b33-2b5f-416d-ad66-9f41d69cbf49)
│   Assigned to: Research Planner
│   Status: ✅ Completed
│   └── 🧠 Memory Update Overall
│       ├── ✅ Short Term Memory Memory Saved (246.47ms)
│       └── ✅ Long Term Memory Memory Saved (1.89ms)
📋 Task: create_research_plan (ID: 147e0b33-2b5f-416d-ad66-9f41d69cbf49)
Assigned to: Research Planner
Status: ✅ Completed
└── 🧠 Memory Update Overall
    ├── ✅ Short Term Memory Memory Saved (246.47ms)
    ├── ✅ Long Term Memory Memory Saved (1.89ms)
📋 Task: create_research_plan (ID: 147e0b33-2b5f-416d-ad66-9f41d69cbf49)
Assigned to: Research Planner
Status: ✅ Completed
└── 🧠 Memory Update Overall
    ├── ✅ Short Term Memory Memory Saved (246.47ms)
    ├── ✅ Long Term Memory Memory Saved (1.89ms)
    └── ✅ Entity Memory Memory Saved (1078.97ms)
╭───────────────────────

📋 Task: create_research_plan (ID: 147e0b33-2b5f-416d-ad66-9f41d69cbf49)
Assigned to: Research Planner
Status: ✅ Completed
├── 🧠 Memory Update Overall
│   ├── ✅ Short Term Memory Memory Saved (246.47ms)
│   ├── ✅ Long Term Memory Memory Saved (1.89ms)
│   └── ✅ Entity Memory Memory Saved (1078.97ms)
└── ✅ Memory Retrieval Completed
    └── Sources Used
        ├── ✅ Long Term Memory (0.81ms)
        ├── ✅ Long Term Memory (0.55ms)
        ├── ✅ Short Term Memory (244.13ms)
📋 Task: create_research_plan (ID: 147e0b33-2b5f-416d-ad66-9f41d69cbf49)
Assigned to: Research Planner
Status: ✅ Completed
├── 🧠 Memory Update Overall
│   ├── ✅ Short Term Memory Memory Saved (246.47ms)
│   ├── ✅ Long Term Memory Memory Saved (1.89ms)
│   └── ✅ Entity Memory Memory Saved (1078.97ms)
├── ✅ Memory Retrieval Completed
│   └── Sources Used
│       ├── ✅ Long Term Memory (0.81ms)
│       ├── ✅ Long Term Memory (0.55ms)
│       ├── ✅ Short Term Memory (244.13ms)
│       └── ✅ Entity Memory (203.96ms)
├── 🔍 K

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**输出停在半路？**（2026-10-21 实测）crew 跑了约 5 分钟，上面这格的输出却停在 `create_research_plan` 附近：项目钉的 crewai 1.3.0 在 notebook 的 `!` 里长时间运行时就会停止刷新，但进程会继续跑完。下面的报告文件和 SQLite 里的 state 时间戳都证明它跑完了；想看完整日志就在终端里跑。

</div>

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

**▶️ 在终端运行**

想自己交互输入（包括回答澄清问题），在 VSCode 终端里跑；结束时会打印 `Trace batch finalized … View here: https://app.crewai.com/...` 的追踪链接，链接被折成两行时要拼起来再打开。

```bash
cd deep_research_flow
crewai run
```

</div>

`save_report_and_summarize` 把完整报告写到本文件夹的 `research_report.md`（路径 `../` 相对于 `deep_research_flow/`）：

In [13]:
from IPython.display import Markdown, display

display(Markdown(Path("research_report.md").read_text()))

# Quality Assessment Report for CrewAI, LangGraph, and AutoGen for Production Multi-Agent Systems

## Executive Summary
This report provides a comprehensive comparison of three leading platforms for building production multi-agent systems: **CrewAI**, **LangGraph**, and **AutoGen**. The analysis delves into their features, limitations, and ideal use cases, providing valuable insights for AI engineers seeking to implement efficient agent architectures. The report sources validated research and offers recommendations based on the unique strengths of each platform.

## 1. Overview of Features
### 1.1 CrewAI
- **Role-based Coordination**: Focuses on organizing agents into specific roles for collaborative tasks.
- **Stateful Workflows**: Supports multi-agent workflows, allowing for dynamic interactions among roles.
- **Use Cases**: Effective for defined tasks in research pipelines and structured business scenarios, such as customer support and sales automation.

### 1.2 LangGraph
- **Complex Workflow Orchestration**: Facilitates explicit state transitions, making it suitable for deterministic processes.
- **Checkpoint System**: Allows for accurate audits and long-term decision-making flows.
- **Ideal for**: Industries where the need for control and auditability are paramount, such as regulatory environments.

### 1.3 AutoGen
- **Lightweight Agent Framework**: Provides a simplified framework to create efficient automated agents.
- **Rapid Development**: Best suited for projects requiring quick iterations without extensive complexity.
- **Application**: Smaller-scale applications or prototypes where agility is prioritized over intricate architecture.

## 2. Limitations of Each Platform
### 2.1 CrewAI
- **Overhead**: Coordination overhead can impact overall performance due to the requirement for well-defined roles.
- **Complexity Management**: May not be as effective in scenarios lacking clearly delineated roles, risking decreased efficiency.

### 2.2 LangGraph
- **Setup Complexity**: Requires significant initial setup efforts, which can deter teams unfamiliar with programming intricacies.
- **High Learning Curve**: May complicate onboarding for users without extensive technical backgrounds.

### 2.3 AutoGen
- **Limited Scalability**: While effective for simpler applications, it may lack extensibility for complex enterprise solutions.
- **Configuration Issues**: Initial setup can lead to problems that might require troubleshooting as user needs grow.

## 3. Use Cases
### 3.1 CrewAI
- **Research Pipelines**: Particularly effective in structures where distinct roles enhance productivity.
- **Business Roles**: Ideal for collaborative environments that involve distinct task assignments.

### 3.2 LangGraph
- **Workflow Needs**: Suited for scenarios necessitating checkpoints and detailed verification processes.
- **Critical Decision-Making**: Commonly used in workflows where audit trails and long-term planning are crucial.

### 3.3 AutoGen
- **Rapid Projects**: Best for simpler applications requiring quick adaptability.
- **Prototype Development**: Allows developers to test concepts quickly without complexity.

## 4. Source Reliability Ratings
This report draws on established AI analysis platforms such as **DataCamp**, **Arize AI**, and credible comparative blogs that offer insights into AI framework capabilities. The information gathered has been verified for consistency and reliability.

## 5. Key Insights and Recommendations
- **Choosing the Right Platform**: Selection should be aligned with the project requirements and team expertise. For roles and workflows requiring structured coordination, CrewAI shines. LangGraph is advantageous for complex, high-control processes, while AutoGen is best for agile and less complex applications.
- **Continuous Learning**: Teams should focus on developing skills relevant to their chosen platform's complexity to optimize process implementation effectively.
- **Monitor Market Trends**: Keeping abreast of advancements will better position teams to leverage emerging technologies and methodologies in agent-based systems.

## Summary
This assessment provides a clear understanding of CrewAI, LangGraph, and AutoGen, enabling stakeholders to make informed decisions on which platform aligns best with their operational needs. Ongoing review of platforms and features is advisable to stay ahead in the rapidly evolving AI landscape.

## Citations
- DataCamp. (2023). CrewAI: Exploring Features and Limitations. Retrieved from [DataCamp Website](https://www.datacamp.com)
- Arize AI. (2023). A Comparative Analysis of AI Frameworks. Retrieved from [Arize AI Blog](https://www.arize.ai/blog)
- WORLD AI GUIDE. (2023). Evaluate & Compare AI Tools. Retrieved from [WORLD AI GUIDE Link](https://world-ai-guide.com)
- Microsoft Research. (2023). Exploring Early Adopters Use of AI-Driven Multi-Agent Systems. Retrieved from [Microsoft Research](https://www.microsoft.com/en-us/research)

This report synthesizes verified data to guide John Doe and other professionals in selecting the most suitable platform for building production multi-agent systems.

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 💾 @persist 存到哪了？

</div>

crewai 默认用 SQLite：`<用户数据目录>/<项目文件夹名>/flow_states.db`，每个方法完成都写一行。直接读出来看：

In [2]:
import json, sqlite3
import appdirs   # crewai 自己就用 appdirs 定位这个目录

db = Path(appdirs.user_data_dir("deep_research_flow", "CrewAI")) / "flow_states.db"
rows = sqlite3.connect(db).execute(
    "SELECT method_name, timestamp, state_json FROM flow_states WHERE flow_uuid = 'our-deep-research_flow' ORDER BY id DESC LIMIT 6"
).fetchall()
for method, ts, state in rows[::-1]:
    s = json.loads(state)
    print(f"{ts[:19]}  {method:26} needs_research={s['needs_research']}  query={s['user_query'][:50]!r}")

2026-10-22T02:59:44  start_conversation         needs_research=False  query='Compare CrewAI, LangGraph and AutoGen for building'
2026-10-22T02:59:45  analyze_query              needs_research=True  query='Compare CrewAI, LangGraph and AutoGen for building'
2026-10-22T02:59:46  clarify_query              needs_research=True  query='Compare CrewAI, LangGraph and AutoGen for building'
2026-10-22T03:04:30  execute_research           needs_research=True  query='Compare CrewAI, LangGraph and AutoGen for building'
2026-10-22T03:04:33  save_report_and_summarize  needs_research=True  query='Compare CrewAI, LangGraph and AutoGen for building'
2026-10-22T03:04:33  return_final_answer        needs_research=True  query='Compare CrewAI, LangGraph and AutoGen for building'


最新一行就是最后的 state：上面输出被截断而没看到的最终回答，也能从这里取出来。

In [3]:
last_state = json.loads(rows[0][2])   # rows 按 id 倒序，第 0 行最新
print(last_state["final_answer"])

This is a summary of the final answer:

The research report evaluates three prominent platforms for building production multi-agent systems: **CrewAI**, **LangGraph**, and **AutoGen**, highlighting their features, limitations, and recommended use cases for AI engineers. **CrewAI** excels in role-based coordination and dynamic multi-agent workflows, making it ideal for research pipelines and structured business environments; however, its coordination overhead can impede performance in less defined scenarios. **LangGraph** offers robust workflow orchestration and a checkpoint system, suitable for industries requiring strict control and auditability, but it suffers from significant setup complexity and a steep learning curve. **AutoGen** presents a lightweight framework for rapid development of simpler applications or prototypes but lacks scalability for more complex needs. The report emphasizes choosing the right platform based on project requirements and team expertise while encouraging

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 坑**

| 坑 | 现象 | 原因 / 对策 |
|---|---|---|
| 在 notebook 里直接 `flow.kickoff()` | `asyncio.run() cannot be called from a running event loop` | 用 `await flow.kickoff_async(...)`；或像本 notebook 一样走 CLI |
| flow 里用 `input()`，在 Jupyter 直接跑 | 卡住或拿不到输入 | 终端里跑，或 `printf '答案\n' \| crewai run` 喂 stdin |
| `@listen("SIMPLE")` 但 router 返回 `"simple"` | 分支永远不触发，flow 静默结束 | 事件名区分大小写，必须和 return 的字符串完全一致 |
| 固定 `id` + `@persist` | 旧 state（如 `needs_research=True`）被带进新一次运行 | 这是特性；需要全新开始就换 id 或不传 id |
| 找不到 `crewai_flow.html` | 项目目录里没有 | 1.3.0 写到临时目录并自动打开浏览器 |
| state 字段不给默认值 | Flow 创建 state 时报 ValidationError | 每个字段都写默认值 |
| crew 的 task 提到 custom tool 画图 | 报告里没有图 | 这个 flow 的 `report_writer` 没配 `ChartGeneratorTool`，提示词说了也调不了 |

</div>

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

**🔁 复习自测**（先自己答，再点开看答案）

<details><summary><b>1. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">@router</code> 和 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">@listen</code> 的区别？</b></summary>

两者都在上游完成后运行；`@router` 的**返回值**会作为事件名再分发，只有 `@listen("该值")` 的方法被触发。`@listen` 的返回值只是这个方法的输出。

</details>

<details><summary><b>2. 为什么 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">return_final_answer</code> 要用 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">or_</code>？</b></summary>

它有两条可能的上游（`simple_answer` 或 `save_report_and_summarize`），一次运行只会走一条；不写 `or_` 只能监听一个。`and_` 则要两者都完成，这里永远等不到。

</details>

<details><summary><b>3. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">@persist()</code> 加在类上会做什么？再次运行时怎样恢复？</b></summary>

每个方法执行完都把 `self.state` 存进 SQLite（`flow_states.db`）。`kickoff(inputs={"id": 同一个 id})` 时先从库里加载这个 id 最新的 state。

</details>

<details><summary><b>4. 这个 flow 在 SIMPLE 路径上一共调了几次 LLM？RESEARCH 路径呢？</b></summary>

SIMPLE：router 1 次 + 直接回答 1 次 = 2 次。RESEARCH：router 1 次 + 澄清 1 次 + 整个 crew（很多次，含搜索）+ 摘要 1 次。

</details>

<details><summary><b>5. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">execute_research</code> 存的是 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">result.raw</code>，<code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">result</code> 是什么？</b></summary>

`crew.kickoff()` 返回的 `CrewOutput`；`.raw` 是最后一个 task（`write_final_report`）的文本输出。

</details>

<details><summary><b>6. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">tracing=True</code> 带来什么？</b></summary>

运行事件（每个方法、agent、工具调用、token）上报到 CrewAI 平台，结束时给出一个可分享的追踪链接，用于监控和排错。

</details>

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：**Flow 用确定的 Python 代码编排、Crew 负责开放的协作**：`@start` → `@router` 按 LLM 判断分流 → `@listen` 串步骤 → `or_` 汇合；`@persist` 存 state，`tracing` 看全过程。

</div>